## Ultrasound classification on the 12cm usb fan.
This notebook contains simple set of scripts to create a proof of the concept anomaly classification model using Convolutional Neural Networks (CNN). The data used in this notebook is connected using the analog microphone on STEVAL-STWINKT1B for a 12CM ELUTENG Fan. The fan can be run at three different speeds, [L = 1000, M = 1250, H = 1500 ] RPMs. The classes included are ['Off', 'Normal', 'Clogging', 'Friction']. 

The script uses a POC using MFCCs but same can be done using spectrogram (STFT), mel-spectrogram or log-mel-spectrogram also.
A simple CNN model definition is provided below in the script but model architecture can be changed. However, it is using a window size of 4096 which is the maximum size supported by the c-implementation, so this has to be respected.

All the audio-processing is done using <b>librosa</b> library.

#### importing required dependencies
We are using UltrasoundDataHelper class for preparing the data, and tensorflow for building the model. Following section imports all the dependencies.

In [1]:
# from UltrasoundDataPreparation_mixed_v3 import UltrasoundDataHelper
# from UltrasoundDataPreparation_mixed_v4_traincol_then_segment_split_smallval_viz import UltrasoundDataHelper
from prepare_command_dataset import UltrasoundDataHelper
from stdatalog_core.HSD.HSDatalog import HSDatalog
print("stdatalog_core OK")

# dataset_path = r'E:\SPEECH_DATA\TENG\command-20260226-ODR16000-2s\DZF-1'
# acq_path = r'E:\SPEECH_DATA\TENG\command-20260226-ODR16000-2s\DZF-1\wav\background\20260228_17_21_45'

# dataset_path = r'E:\SPEECH_DATA\TENG\touming-dataset\command-20260305-ODR16000-1600s\merge-2'
# acq_path = r'E:\SPEECH_DATA\TENG\touming-dataset\command-20260305-ODR16000-1600s\merge-2\wav-4\background\20260306_21_30_20'

# dataset_path = r'E:\SPEECH_DATA\TENG\touming-dataset\command-20260312-ODR16000-2000s'
# acq_path = r'E:\SPEECH_DATA\TENG\touming-dataset\command-20260312-ODR16000-2000s\dat\background\20260312_23_27_53'

dataset_path = r'E:\SPEECH_DATA\TENG\touming-dataset\command-20260419-ODR16000-2s'
acq_path = r'E:\SPEECH_DATA\TENG\touming-dataset\command-20260419-ODR16000-2s\wav\background\20260419_12_09_01'


udh = UltrasoundDataHelper(
    dataset=dataset_path,
    sensor_name='imp23absu',
    sensor_type='mic',
    offset=0,
    sample_rate=16000,
    # frame_len=512 * 52 + 1,
    frame_len=512 * 60 + 1,
    n_fft=512,
    hop_length=512,
    n_mels=32,
    n_mfccs=32,
    # classes=['Fanoff', 'Fanon', 'humidifieroff', 'humidifieron', 'lightoff', 'Lighton', 'radiooff', 'radioon', 'TVoff', 'TVon', 'background'],
    # classes=['Fanoff', 'Fanon', 'humidifieroff', 'humidifieron', 'Lightoff', 'Lighton', 'TVoff', 'TVon', 'background'],
    # classes = ['dehumidifieroff', 'dehumidifieron', 'fanoff', 'fanon', 'humidifieroff', 'humidifieron', 'lightoff', 'lighton', 'Televisionoff', 'Televisionon', 'background'],
    classes = ['microwaveoff', 'microwaveon', 'fanoff', 'fanon', 'humidifieroff', 'humidifieron', 'lightoff', 'lighton', 'Televisionoff', 'Televisionon', 'background'],
    debug=True
)

x = udh.read_hsd_ultrasound(acq_path, 'imp23absu', 'mic', 0)
print(type(x))
print(None if x is None else len(x))
print(None if x is None else x[:10])



stdatalog_core OK
[WAV] E:\SPEECH_DATA\TENG\touming-dataset\command-20260419-ODR16000-2s\wav\background\20260419_12_09_01\imp23absu_mic.wav | n=31000 | offset=0
<class 'numpy.ndarray'>
31000
[ 7 10  9 11 11 10 11  8 11 10]


In [2]:
import numpy, scipy
print("numpy", numpy.__version__)
print("scipy", scipy.__version__)

from stdatalog_core.HSD.HSDatalog import HSDatalog
print("stdatalog_core OK")


numpy 1.26.4
scipy 1.11.4
stdatalog_core OK


In [3]:
%load_ext autoreload
%autoreload 2

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from datetime import datetime
# from UltrasoundDataPreparation import UltrasoundDataHelper
# from UltrasoundDataPreparation_mixed_v3 import UltrasoundDataHelper
# from UltrasoundDataPreparation_mixed_v4 import UltrasoundDataHelper
# from UltrasoundDataPreparation_mixed_v4_segment_split_minimal import UltrasoundDataHelper
from UltrasoundDataPreparation_mixed_v4_traincol_then_segment_split_smallval_viz import UltrasoundDataHelper
# from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# setting random seeds for reproducing
np.random.seed(611)
tf.random.set_seed(611)


### Building an object of the class DataHelper
This section creates a result directory to save the results of the current run using current time as part of the name. It also saves the configurations as ```info.txt``` file in the created result folder using the variables defined in the section.

In [4]:
# dataset_path = '../../Datasets/HSD_Logged_Data/Fan12CM/ism330dhcx+imp23absu/'
# dataset_path = 'E:/SPEECH_DATA/20251029-command/'
# dataset_path = 'E:\SPEECH_DATA\TENG\command-20251202-ODR16000/'
# dataset_path = 'D:\speech_data\command-20251207-ODR16000-2s/'
# dataset_path = 'E:/SPEECH_DATA/TENG\command-20260226-ODR16000-2s/DZF-1/'
# dataset_path = 'E:/SPEECH_DATA/TENG/command-20260302-ODR16000-1600ms/'
# dataset_path = 'E:/SPEECH_DATA/TENG/touming-dataset/command-20260305-ODR16000-1600s/merge-2/'
# dataset_path = 'E:/SPEECH_DATA/TENG/touming-dataset/command-20260312-ODR16000-2000s/'
# dataset_path = 'E:/SPEECH_DATA/TENG/touming-dataset/command-20260314-ODR16000-2000s/'
# dataset_path = 'E:/SPEECH_DATA/TENG/touming-dataset/command-20260315-merge/'
# dataset_path = 'E:/SPEECH_DATA/TENG/touming-dataset/command-20260413-ODR16000-2s/'
dataset_path = 'E:/SPEECH_DATA/TENG/touming-dataset/command-20260419-ODR16000-2s/'

# sensor attributes
sensor_name = 'imp23absu'
sensor_type = 'mic'
sample_rate = 16000
# sample_start = 192000
sample_start = 0

# data preparation configurations
# frame_length = 4096 * 46 + 1  # each inference will use this many samples
# frame_length = 4096 * 88 + 1  # each inference will use this many samples
# frame_length = 4096 * 2 + 1   # each inference will use this many samples
# frame_length = 1024 * 8 + 1   # each inference will use this many samples
# frame_length = 1024 * 26 + 1  # each inference will use this many samples
# frame_length = 512 * 52 + 1     # each inference will use this many samples
frame_length = 512 * 60 + 1     # each inference will use this many samples
# frame_length = 512 * 38 + 1     # each inference will use this many samples
# frame_length = 4096 * 6 + 1   # each inference will use this many samples
window = 'hann'
# n_fft = 4096
# hop_length = 4096  # n_fft = hop_length means no overlap
# n_fft = 1024
# hop_length = 1024  # n_fft = hop_length means no overlap
n_fft = 512
hop_length = 512     # n_fft = hop_length means no overlap
n_mels = 32
n_mfccs = 32

# available classes in the dataset
# classes = ['off', 'normal', 'clogging', 'friction' ]
# classes = ['CloseCurtain', 'OpenCurtain', 'TVoff', 'TVon' ]
# classes = ['lightoff', 'lighton', 'playmusic', 'stopmusic' ]
# classes = ['background', 'lightoff', 'lighton', 'playmusic', 'stopmusic' ]
# classes = ['lightoff', 'lighton', 'switch_off_the_tv', 'switch_on_the_tv' ]
# classes = ['Fanoff', 'Fanon', 'humidifieroff', 'humidifieron', 'lightoff', 'Lighton', 'radiooff', 'radioon', 'TVoff', 'TVon', 'background']

# classes = ['Fanoff', 'Fanon', 'humidifieroff', 'humidifieron', 'Lightoff', 'Lighton', 'TVoff', 'TVon', 'background']
# classes = ['dehumidifieroff', 'dehumidifieron', 'fanoff', 'fanon', 'humidifieroff', 'humidifieron', 'lightoff', 'lighton', 'Televisionoff', 'Televisionon', 'background']
classes = ['microwaveoff', 'microwaveon', 'fanoff', 'fanon', 'humidifieroff', 'humidifieron', 'lightoff', 'lighton', 'Televisionoff', 'Televisionon', 'background']

# automatic train/test split configuration
# 0.8 = 80% train / 20% test
# 0.6 = 60% train / 40% test
auto_split_train_ratio = 0.8
print_file_segment_report = True

# neural network training configurations
epochs = 50
batch_size = 32
val_split = 0.2

# creating a folder for saving the results and the configurations
running_time = datetime.now().strftime("%Y_%b_%d_%H_%M_%S")
# resultDir = os.path.join('results', running_time)
resultDir = './results/' + running_time + '/'

if not os.path.exists(resultDir):
    os.makedirs(resultDir)

infoString = '''runTime : {}

Database : {}
SeqLength : {}
n_fft : {}
hop_length : {}
n_mels : {}
n_mfccs : {}
classes : {}
auto_split_train_ratio : {}
print_file_segment_report : {}
Epochs : {}
batch_size : {}
val_split : {}'''.format(
    datetime.now().strftime("%Y-%b-%d at %H:%M:%S"),
    dataset_path,
    frame_length,
    n_fft,
    hop_length,
    n_mels,
    n_mfccs,
    classes,
    auto_split_train_ratio,
    print_file_segment_report,
    epochs,
    batch_size,
    val_split
)

with open(os.path.join(resultDir, 'info.txt'), 'w') as text_file:
    text_file.write(infoString)

# creating the data helper object
myUDH = UltrasoundDataHelper(
    dataset=dataset_path,
    sensor_name=sensor_name,
    sensor_type=sensor_type,
    offset=sample_start,
    sample_rate=sample_rate,
    frame_len=frame_length,
    n_fft=n_fft,
    hop_length=hop_length,
    n_mels=n_mels,
    n_mfccs=n_mfccs,
    classes=classes,
    debug=True,
    auto_split_train_ratio=auto_split_train_ratio,
    print_file_segment_report=print_file_segment_report
)


In [5]:
import sys
import soundfile as sf
import librosa as lb
import numpy as np
print(sys.executable)
# wav = r"E:\SPEECH_DATA\20251029-command\wav\CloseCurtain\20251021_10_47_37\imp23absu_mic.wav"  # 随便挑一个实际存在的
# wav = r"E:\SPEECH_DATA\TENG\command-20260226-ODR16000-2s\DZF-1\wav\Fanoff\20260226_17_04_12\imp23absu_mic.wav"  # 随便挑一个实际存在的
wav = r"E:\SPEECH_DATA\TENG\touming-dataset\command-20260413-ODR16000-2s\wav\fanoff\20260412_19_54_14\imp23absu_mic.wav"  # 随便挑一个实际存在的

x, sr = sf.read(wav, dtype="int16", always_2d=True)
x = x.mean(axis=1).astype("int16", copy=False)  # 转单声道
print("len:", len(x), "sr:", sr)
mfcc = lb.feature.mfcc(y=(x.astype(np.float32)/32768.0), sr=sr, n_mfcc=13)
print("MFCC shape:", mfcc.shape)


d:\workspace\project\keil\thesis_speech_recognition\usc_train\Scripts\python.exe
len: 29000 sr: 16000
MFCC shape: (13, 57)


In [6]:
import numpy as np
import sys
print(sys.executable)
print("numpy =", np.__version__, "\n", np.__file__)


d:\workspace\project\keil\thesis_speech_recognition\usc_train\Scripts\python.exe
numpy = 1.26.4 
 d:\workspace\project\keil\thesis_speech_recognition\usc_train\lib\site-packages\numpy\__init__.py


### Creating the input data and labels
This section uses the myUDH object of ```UltrasoundDataHelper``` to create the input feature matrices and labels to feed to neural networks. 

It also prints the shapes of the input data and labels for both train and test data.

In [ ]:
mfccsTrain, labelsTrain, mfccsTest, labelsTest = myUDH.get_data_mfccs()

for name, arr in [
    ("mfccsTrain", mfccsTrain),
    ("mfccsTest", mfccsTest),
    ("labelsTrain", labelsTrain),
    ("labelsTest", labelsTest),
]:
    print(name, type(arr), getattr(arr, "shape", None))

print("train count per class:", np.bincount(np.argmax(labelsTrain, axis=1), minlength=len(classes)))
print("test count per class:", np.bincount(np.argmax(labelsTest, axis=1), minlength=len(classes)))

# Save per-acquisition segment report to CSV for later checking
if hasattr(myUDH, "last_file_segment_report") and myUDH.last_file_segment_report is not None:
    report_df = myUDH.last_file_segment_report.copy()
    report_csv = os.path.join(resultDir, "file_segment_report.csv")
    report_df.to_csv(report_csv, index=False, encoding="utf-8-sig")
    print(f"Saved per-acquisition report to: {report_csv}")

    print("\n=== Per-acquisition segment report (all files) ===")
    print(report_df[["label", "display_name", "split", "source_kind", "segments", "status"]].to_string(index=False))

    print("\n=== Background only ===")
    bg_df = report_df[report_df["label"].astype(str).str.lower() == "background"].copy()
    if len(bg_df):
        print(bg_df[["display_name", "split", "source_kind", "segments", "status"]].to_string(index=False))
    else:
        print("No background rows found in file_segment_report.")
else:
    print("No file-segment report found on myUDH.")


[SPLIT] metadata.csv 'train' column is present and all rows are TRAIN; keeping all acquisitions in TRAIN first. If TEST stays empty after feature extraction, a segment-level split will be applied later.
[WAV] E:\SPEECH_DATA\TENG\touming-dataset\command-20260419-ODR16000-2s\wav\Televisionoff\20260419_14_19_36\imp23absu_mic.wav | n=31000 | offset=0
[WAV] E:\SPEECH_DATA\TENG\touming-dataset\command-20260419-ODR16000-2s\wav\Televisionoff\20260419_14_19_39\imp23absu_mic.wav | n=31000 | offset=0
[WAV] E:\SPEECH_DATA\TENG\touming-dataset\command-20260419-ODR16000-2s\wav\Televisionoff\20260419_14_19_43\imp23absu_mic.wav | n=31000 | offset=0
[WAV] E:\SPEECH_DATA\TENG\touming-dataset\command-20260419-ODR16000-2s\wav\Televisionoff\20260419_14_19_46\imp23absu_mic.wav | n=31000 | offset=0
[WAV] E:\SPEECH_DATA\TENG\touming-dataset\command-20260419-ODR16000-2s\wav\Televisionoff\20260419_14_19_49\imp23absu_mic.wav | n=31000 | offset=0
[WAV] E:\SPEECH_DATA\TENG\touming-dataset\command-20260419-ODR16000

In [ ]:
# Reuse the data generated in the previous cell
mfccsTrain = np.transpose(mfccsTrain, [0, 2, 1, 3])  # to match the shape in the C implementation
mfccsTest = np.transpose(mfccsTest, [0, 2, 1, 3])    # to match the shape in the C implementation

# printing the shapes of the data
print('Shape of train data : ', mfccsTrain.shape)
print('Shape of train labels : ', labelsTrain.shape)
print('Shape of test data : ', mfccsTest.shape)
print('Shape of test labels : ', labelsTest.shape)


In [ ]:
# optional
# saving the data as npz file to avoid the data preparation steps again

np.savez( resultDir + 'train_valid_data.npz', mfccsTest = mfccsTest, labelsTest = labelsTest, mfccsTrain = mfccsTrain, labelsTrain = labelsTrain)


### Building the network
Following section defines a simple network for the anomaly classification. 

In [ ]:
# def build_model_4096( inputShape, nrClasses ):
#     model = tf.keras.models.Sequential([
#         tf.keras.Input( shape = inputShape ),
#         tf.keras.layers.Lambda(lambda x : x[ :, :, 1:, :]),
#         tf.keras.layers.Conv2D( 16, ( 5, 5 ), input_shape = inputShape, activation = 'relu' ),
#         tf.keras.layers.BatchNormalization(),
#         tf.keras.layers.MaxPool2D( pool_size = ( 5, 5 ) ),
#         tf.keras.layers.Flatten(),
#         tf.keras.layers.Dropout(0.25),
#         tf.keras.layers.Dense(64, activation = 'relu' ),
#         tf.keras.layers.Dropout(0.25),
#         tf.keras.layers.Dense(nrClasses, activation= 'softmax')
#         ])
#     adam = tf.keras.optimizers.Adam( lr = 0.001 )
#     model.compile( loss = 'categorical_crossentropy', optimizer = adam, metrics =['acc'])
#     return model

# def build_model_4096( inputShape, nrClasses ):
#     model = tf.keras.models.Sequential([
#         tf.keras.Input( shape = inputShape ),
#         tf.keras.layers.Lambda(lambda x : x[ :, :, 1:, :]),
#         tf.keras.layers.Conv2D( 8, ( 5, 5 ), input_shape = inputShape, activation = 'relu' ),
#         tf.keras.layers.BatchNormalization(),
#         tf.keras.layers.MaxPool2D( pool_size = ( 5, 5 ) ),
#         # tf.keras.layers.MaxPool2D( pool_size = ( 4, 4 ) ),
#         # tf.keras.layers.MaxPool2D( pool_size = ( 2, 2 ) ),
#         tf.keras.layers.Flatten(),
#         tf.keras.layers.Dropout(0.25),
#         tf.keras.layers.Dense(64, activation = 'relu' ),
#         tf.keras.layers.Dropout(0.25),
#         tf.keras.layers.Dense(nrClasses, activation= 'softmax')
#         ])
#     adam = tf.keras.optimizers.Adam( learning_rate= 0.001 )
#     model.compile( loss = 'categorical_crossentropy', optimizer = adam, metrics =['acc'])
#     return model

# def build_model_4096( inputShape, nrClasses ):
#     model = tf.keras.models.Sequential([
#         tf.keras.Input( shape = inputShape ),
#         tf.keras.layers.Lambda(lambda x : x[ :, :, 1:, :]),
#         tf.keras.layers.Conv2D(16, (5, 5), activation='relu'),
#         tf.keras.layers.BatchNormalization(),
#         tf.keras.layers.MaxPool2D(pool_size=(5, 5)),
#         tf.keras.layers.Flatten(),
#         tf.keras.layers.Dropout(0.20),
#         tf.keras.layers.Dense(128, activation='relu'),
#         tf.keras.layers.Dropout(0.20),
#         tf.keras.layers.Dense(nrClasses, activation= 'softmax')
#         ])
#     adam = tf.keras.optimizers.Adam( learning_rate= 0.001 )
#     model.compile( loss = 'categorical_crossentropy', optimizer = adam, metrics =['acc'])
#     return model

def build_model_4096( inputShape, nrClasses ):
    model = tf.keras.models.Sequential([
        tf.keras.Input( shape = inputShape ),
        tf.keras.layers.Lambda(lambda x : x[ :, :, 1:, :]),

        tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPool2D(pool_size=(2, 2)),

        tf.keras.layers.Conv2D(24, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPool2D(pool_size=(2, 2)),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.20),

        # tf.keras.layers.Dense(128, activation='relu'),
        # 关键改动：128 -> 32
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Dense(nrClasses, activation='softmax')

        ])
    adam = tf.keras.optimizers.Adam( learning_rate= 0.001 )
    model.compile( loss = 'categorical_crossentropy', optimizer = adam, metrics =['acc'])
    return model



In [ ]:
inputShape = ( mfccsTrain.shape[1], mfccsTrain.shape[2], mfccsTrain.shape[3] )
nrClasses = labelsTrain.shape[1]
# model = build_model_4096( inputShape=inputShape, nrClasses = nrClasses )
model = build_model_4096( inputShape= inputShape, nrClasses = nrClasses)
model.summary()


### Training the network

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=8,
    restore_best_weights=True,
    mode='max'
)


In [ ]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-5,
    verbose=1
)


In [ ]:
model_name = resultDir + 'model.h5' 
# history = model.fit( melsTrain, labelsTrain, batch_size = 64, validation_split = 0.3, epochs = 100, verbose = 2, callbacks = usc_model_checkpoint )
# history = model.fit( mfccsTrain, labelsTrain, batch_size = batch_size, validation_split = 0.2, epochs = epochs, verbose = 2 )
history = model.fit(
    mfccsTrain,
    labelsTrain,
    batch_size=batch_size,
    validation_split=0.2,
    epochs=epochs,
    verbose=2,
    callbacks=[early_stop, reduce_lr]
)


### Visualising the evolution of the training accuracy and loss vs epochs

In [ ]:
plt.figure( figsize = (20,10))
plt.subplot(1,2,1)
plt.plot(history.history[ 'acc' ])
plt.plot(history.history[ 'val_acc' ])
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend( ['Train', 'Validation' ])
plt.savefig(resultDir + 'accuracy_history.png')
plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend( ['Train', 'Validation' ])
plt.savefig(resultDir + 'loss_history.png')


### Evaluating the trained model

In [ ]:
scoreTrain = model.evaluate(mfccsTrain, labelsTrain)
scoreTrain = model.evaluate(mfccsTest, labelsTest)
print('Train Confusion matrix')
print( confusion_matrix( np.argmax(labelsTrain, axis = 1), np.argmax( model.predict( mfccsTrain ), axis = 1 ) ) )
print('Test Confusion matrix')
print( confusion_matrix( np.argmax(labelsTest, axis = 1), np.argmax( model.predict( mfccsTest ), axis = 1 ) ) )


### Converting the trained model into a tflite model

In [ ]:
# model = tf.keras.models.load_model(model_name)
converter = tf.lite.TFLiteConverter.from_keras_model(model) # path to the SavedModel directory
tflite_model = converter.convert()

# Save the model.
with open('{}.tflite'.format(model_name[:-3]), 'wb') as f:
  f.write(tflite_model)


In [ ]:
import numpy as np
from pathlib import Path

# 训练时生成的总数据文件
src = Path(resultDir) / "train_valid_data.npz"

# 输出到当前结果目录，和 model.tflite 放一起
dst = Path(resultDir) / "post_validation_data.npz"

data = np.load(src)

print("Available keys:", data.files)

x_test = data["mfccsTest"].astype(np.float32)
y_test = data["labelsTest"].astype(np.float32)

print("x_test shape:", x_test.shape, x_test.dtype)
print("y_test shape:", y_test.shape, y_test.dtype)

np.savez(dst, x_test=x_test, y_test=y_test)

print(f"Saved to: {dst}")


In [ ]:
y_true = np.argmax(labelsTest, axis=1)
y_pred = np.argmax(model.predict(mfccsTest), axis=1)

print("classes:", len(classes), classes)
print("unique y_true:", np.unique(y_true))
print("unique y_pred:", np.unique(y_pred))
print("test count per class:", np.bincount(y_true, minlength=len(classes)))


In [ ]:
x = udh.read_hsd_ultrasound(acq_path, 'imp23absu', 'mic', 0)
print(type(x))
print(None if x is None else len(x))


In [ ]:


dataset_path = r'E:\SPEECH_DATA\TENG\command-20260226-ODR16000-2s\DZF-1'
acq_path = r'E:\SPEECH_DATA\TENG\command-20260226-ODR16000-2s\DZF-1\wav\background\20260210_17_29_25'

udh = UltrasoundDataHelper(
    dataset=dataset_path,
    sensor_name='imp23absu',
    sensor_type='mic',
    offset=0,
    sample_rate=16000,
    frame_len=512 * 52 + 1,
    n_fft=512,
    hop_length=512,
    n_mels=32,
    n_mfccs=32,
    classes=['Fanoff', 'Fanon', 'humidifieroff', 'humidifieron', 'lightoff', 'Lighton', 'radiooff', 'radioon', 'TVoff', 'TVon', 'background'],
    debug=True
)

x = udh.read_hsd_ultrasound(acq_path, 'imp23absu', 'mic', 0)
print(type(x))
print(None if x is None else len(x))


In [ ]:
print("resultDir =", resultDir)
import os
print("cwd =", os.getcwd())


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# =========================
# 0) 固定导出目录
# =========================
out_dir = Path(resultDir).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
print("Export dir:", out_dir)

# =========================
# 1) 导出训练曲线原始数据（history）
# =========================
if "history" not in globals():
    raise RuntimeError("Variable `history` not found. Please run the model.fit(...) cell first.")
if not hasattr(history, "history"):
    raise RuntimeError("`history` exists but has no `.history` attribute. Check your training cell.")

hist = history.history

# 1.1 CSV
hist_df = pd.DataFrame(hist)

# ✅ 增加 epoch 横坐标（1-based）
if "epoch" not in [c.lower() for c in hist_df.columns]:
    # hist_df.insert(0, "epoch", range(1, len(hist_df) + 1))
    hist_df.insert(0, "epoch", range(len(hist_df)))
p_csv = out_dir / "train_history.csv"
hist_df.to_csv(p_csv, index=False, encoding="utf-8-sig")
print("Saved:", p_csv)

# 1.2 JSON（你要求的 hist_clean 写法：把 numpy.float32 变成 float）
hist_clean = {}
for k, v in hist.items():
    # v 通常是 list[float]，但元素可能是 numpy.float32
    hist_clean[k] = [float(x) for x in v]

p_json = out_dir / "train_history.json"
with open(p_json, "w", encoding="utf-8") as f:
    json.dump(hist_clean, f, ensure_ascii=False, indent=2)
print("Saved:", p_json)

# 1.3 NPZ（保留 numpy array 形式）
p_npz = out_dir / "train_history.npz"
np.savez(p_npz, **{k: np.array(v) for k, v in hist.items()})
print("Saved:", p_npz)

# =========================
# 2) 从原始数据重画并保存 accuracy/loss 曲线
# =========================
# 2.1 accuracy
if "accuracy" in hist_clean or "val_accuracy" in hist_clean:
    plt.figure()
    if "accuracy" in hist_clean:
        plt.plot(hist_clean["accuracy"], label="Train")
    if "val_accuracy" in hist_clean:
        plt.plot(hist_clean["val_accuracy"], label="Validation")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    p = out_dir / "accuracy_history_from_data.png"
    plt.savefig(p, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved:", p)

# 2.2 loss
if "loss" in hist_clean or "val_loss" in hist_clean:
    plt.figure()
    if "loss" in hist_clean:
        plt.plot(hist_clean["loss"], label="Train")
    if "val_loss" in hist_clean:
        plt.plot(hist_clean["val_loss"], label="Validation")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    p = out_dir / "loss_history_from_data.png"
    plt.savefig(p, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved:", p)

# =========================
# 3) 导出混淆矩阵
# =========================
required = ["mfccsTest", "labelsTest", "model", "classes"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing variables for confusion matrix: {missing}. Please run evaluation cells first.")

y_true = np.argmax(labelsTest, axis=1)
y_pred = np.argmax(model.predict(mfccsTest), axis=1)

cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(classes)))

# 3.1 CSV（带行列标签）
p_cm_csv = out_dir / "confusion_matrix.csv"
pd.DataFrame(cm, index=classes, columns=classes).to_csv(p_cm_csv, encoding="utf-8-sig")
print("Saved:", p_cm_csv)

# 3.2 NPY
p_cm_npy = out_dir / "confusion_matrix.npy"
np.save(p_cm_npy, cm)
print("Saved:", p_cm_npy)

# 3.3 PNG + PDF（PDF 便于论文矢量）
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", values_format="d")
plt.tight_layout()

p_cm_png = out_dir / "confusion_matrix.png"
p_cm_pdf = out_dir / "confusion_matrix.pdf"
plt.savefig(p_cm_png, dpi=300, bbox_inches="tight")
plt.savefig(p_cm_pdf, bbox_inches="tight")
plt.close()
print("Saved:", p_cm_png)
print("Saved:", p_cm_pdf)

print("All exports done.")


### Evaluating using a single recording

In [ ]:
import sys
print(sys.executable)


In [ ]:
# # mfccs, labels = myUDH.get_data_mfcc_single_acquisition( acq_path = '../../Datasets/HSD_Logged_Data/Fan12CM/ISM330DHCX+IMP23ABSU/Off', label = 'off' )
# wav = r"E:/SPEECH_DATA/20251029-command/wav/CloseCurtain/20251021_10_47_37/imp23absu_mic.wav"
# mfccs, labels = myUDH.get_data_mfcc_single_acquisition(acq_path=wav, label='CloseCurtain')
# print( 'accuracy on CloseCurtain data is ', np.round( model.evaluate( mfccs.transpose([0,2,1,3]), labels, verbose = 0)[1] * 100, 2 ), '%' )

# # mfccs, labels = myUDH.get_data_mfcc_single_acquisition( acq_path = '../../Datasets/HSD_Logged_Data/Fan12CM/ISM330DHCX+IMP23ABSU/Normal', label = 'normal' )

# wav = r"E:/SPEECH_DATA/20251029-command/wav/OpenCurtain/20251021_10_34_49/imp23absu_mic.wav"
# mfccs, labels = myUDH.get_data_mfcc_single_acquisition(acq_path=wav, label='OpenCurtain')
# print( 'accuracy on OpenCurtain data is ', np.round( model.evaluate( mfccs.transpose([0,2,1,3]), labels, verbose = 0)[1] * 100, 2 ), '%' )

# # mfccs, labels = myUDH.get_data_mfcc_single_acquisition( acq_path = '../../Datasets/HSD_Logged_Data/Fan12CM/ISM330DHCX+IMP23ABSU/Clogging', label = 'clogging' )

# wav = r"E:/SPEECH_DATA/20251029-command/wav/TVoff/20251021_11_10_15/imp23absu_mic.wav"
# mfccs, labels = myUDH.get_data_mfcc_single_acquisition(acq_path=wav, label='TVoff')
# print( 'accuracy on TVoff data is ', np.round( model.evaluate( mfccs.transpose([0,2,1,3]), labels, verbose = 0)[1] * 100, 2 ), '%' )

# # mfccs, labels = myUDH.get_data_mfcc_single_acquisition( acq_path = '../../Datasets/HSD_Logged_Data/Fan12CM/ISM330DHCX+IMP23ABSU/Friction', label = 'friction' )
# wav = r"E:/SPEECH_DATA/20251029-command/wav/TVon/20251021_10_59_10/imp23absu_mic.wav"
# mfccs, labels = myUDH.get_data_mfcc_single_acquisition(acq_path=wav, label='TVon')
# print( 'accuracy on TVon data is ', np.round( model.evaluate( mfccs.transpose([0,2,1,3]), labels, verbose = 0)[1] * 100, 2 ), '%' )


In [ ]:
# # === Export: Confusion matrix + raw history data (accuracy/loss) ===
# import os, json
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# # 1) Resolve output directory (reuse resultDir if defined)
# _out_dir = globals().get("resultDir", os.getcwd())
# os.makedirs(_out_dir, exist_ok=True)

# # 2) Grab Keras history object (try common variable names)
# _hist_obj = None
# for _name in ["history", "hist", "training_history", "trainingHistory", "H"]:
#     if _name in globals():
#         _hist_obj = globals()[_name]
#         break

# # 3) Export training curves raw data
# if _hist_obj is not None and hasattr(_hist_obj, "history"):
#     _h = _hist_obj.history
#     # Build a table by epoch index (1..N)
#     _epochs = list(range(1, len(next(iter(_h.values()))) + 1)) if len(_h) else []
#     _df = pd.DataFrame({"epoch": _epochs})
#     for _k, _v in _h.items():
#         _df[_k] = _v
#     _csv_path = os.path.join(_out_dir, "train_history.csv")
#     _json_path = os.path.join(_out_dir, "train_history.json")
#     _npz_path = os.path.join(_out_dir, "train_history.npz")
#     _df.to_csv(_csv_path, index=False, encoding="utf-8-sig")
#     with open(_json_path, "w", encoding="utf-8") as _f:
#         json.dump(_h, _f, ensure_ascii=False, indent=2)
#     np.savez_compressed(_npz_path, **{k: np.array(v) for k, v in _h.items()})
#     print(f"[OK] Saved history: {_csv_path}")
#     print(f"[OK] Saved history: {_json_path}")
#     print(f"[OK] Saved history: {_npz_path}")

#     # Also regenerate and save the two figures from raw data
#     # Accuracy plot
#     if ("accuracy" in _h or "acc" in _h) and ("val_accuracy" in _h or "val_acc" in _h):
#         _acc_key = "accuracy" if "accuracy" in _h else "acc"
#         _vacc_key = "val_accuracy" if "val_accuracy" in _h else "val_acc"
#         plt.figure()
#         plt.plot(_h[_acc_key], label="Train")
#         plt.plot(_h[_vacc_key], label="Validation")
#         plt.xlabel("Epochs")
#         plt.ylabel("Accuracy")
#         plt.legend()
#         _acc_fig = os.path.join(_out_dir, "accuracy_history.png")
#         plt.savefig(_acc_fig, dpi=300, bbox_inches="tight")
#         plt.close()
#         print(f"[OK] Saved figure: {_acc_fig}")

#     # Loss plot
#     if ("loss" in _h) and ("val_loss" in _h):
#         plt.figure()
#         plt.plot(_h["loss"], label="Train")
#         plt.plot(_h["val_loss"], label="Validation")
#         plt.xlabel("Epochs")
#         plt.ylabel("Loss")
#         plt.legend()
#         _loss_fig = os.path.join(_out_dir, "loss_history.png")
#         plt.savefig(_loss_fig, dpi=300, bbox_inches="tight")
#         plt.close()
#         print(f"[OK] Saved figure: {_loss_fig}")
# else:
#     print("[WARN] No Keras History object found. Skipped exporting training curves.")

# # 4) Export confusion matrix (raw + figure)
# # Requires: labelsTest, mfccsTest, model, classes
# _required = ["labelsTest", "mfccsTest", "model", "classes"]
# _missing = [x for x in _required if x not in globals()]
# if _missing:
#     print(f"[WARN] Missing variables for confusion matrix export: {_missing}")
# else:
#     y_true = np.argmax(labelsTest, axis=1)
#     y_pred = np.argmax(model.predict(mfccsTest), axis=1)
#     labels = np.arange(len(classes))
#     cm = confusion_matrix(y_true, y_pred, labels=labels)

#     cm_csv = os.path.join(_out_dir, "confusion_matrix.csv")
#     cm_npy = os.path.join(_out_dir, "confusion_matrix.npy")
#     pd.DataFrame(cm, index=classes, columns=classes).to_csv(cm_csv, encoding="utf-8-sig")
#     np.save(cm_npy, cm)
#     print(f"[OK] Saved confusion matrix: {cm_csv}")
#     print(f"[OK] Saved confusion matrix: {cm_npy}")

#     disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
#     fig, ax = plt.subplots(figsize=(10, 10))
#     disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", values_format="d")
#     plt.title("Confusion Matrix")
#     cm_png = os.path.join(_out_dir, "confusion_matrix.png")
#     plt.savefig(cm_png, dpi=300, bbox_inches="tight")
#     plt.close(fig)
#     print(f"[OK] Saved confusion matrix figure: {cm_png}")


## Visual export utilities for 11 classes — fixed version

This fixed section does **not** search folders by guessing class names.  
It reuses `myUDH.last_file_segment_report`, which was produced by the same data-loading path used for training, so it should export all 11 classes as long as they appeared in the training/test data.

Output folder:

```python
os.path.join(resultDir, 'visual_exports_fixed')
```

Generated outputs:

- `waveforms_11classes/` — one waveform image per class
- `logmel_11classes/` — one Log-Mel image per class
- `mfcc_11classes/` — one MFCC image per class
- `architecture/` — 3D-style model framework PNG/PDF
- `visual_export_summary.csv`


In [ ]:
# ============================================================
# Fixed visual export block
# Uses myUDH.last_file_segment_report instead of guessing folders
# ============================================================
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import librosa as lb
from matplotlib.patches import Rectangle, FancyArrowPatch

import matplotlib.colors as mcolors

# MFCC 专用高对比配色：深绿色背景 → 青绿色 → 亮蓝色高亮
MFCC_HIGH_CONTRAST_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "MFCC_HighContrast_GreenBlue",
    [
        (0.00, "#002b17"),
        (0.22, "#006d3c"),
        (0.50, "#18c77a"),
        (0.75, "#1cc9ff"),
        (1.00, "#003cff"),
    ],
    N=256,
)

# 百分位裁剪增强对比度，避免极端值把整张图压得很淡
MFCC_CONTRAST_LOW_PERCENTILE = 1
MFCC_CONTRAST_HIGH_PERCENTILE = 99


import matplotlib.colors as mcolors

# MFCC 专用配色：绿色背景，蓝色高亮
MFCC_GREEN_BLUE_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "MFCC_GreenBackground_BlueHighlight",
    [
        (0.00, "#003b1f"),  # deep green background
        (0.35, "#0b7a3b"),  # green
        (0.65, "#4ccf7a"),  # light green transition
        (1.00, "#0057ff"),  # blue highlight
    ],
    N=256,
)


FIG_W_CM = 3.0
FIG_H_CM = 2.0
FIG_DPI = 300


def cm_to_inch(cm):
    return cm / 2.54


def _ensure_dir(p):
    Path(p).mkdir(parents=True, exist_ok=True)


def _pad_or_crop(signal, target_len):
    signal = np.asarray(signal)
    if len(signal) >= target_len:
        return signal[:target_len]
    out = np.zeros(target_len, dtype=signal.dtype)
    out[:len(signal)] = signal
    return out


def _safe_filename(name):
    return str(name).replace("/", "_").replace("\\", "_").replace(":", "_").replace(" ", "_")


def _compute_logmel(segment, sample_rate, n_fft, hop_length, n_mels):
    y = segment.astype(np.float32)
    if y.size and np.max(np.abs(y)) > 1.0:
        y = y / 32768.0
    mel = lb.feature.melspectrogram(
        y=y,
        sr=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=n_fft,
        n_mels=n_mels,
        center=False,
        power=2.0,
    )
    return lb.power_to_db(mel, ref=np.max)


def _compute_mfcc(segment, sample_rate, n_fft, hop_length, n_mels, n_mfccs):
    y = segment.astype(np.float32)
    if y.size and np.max(np.abs(y)) > 1.0:
        y = y / 32768.0
    return lb.feature.mfcc(
        y=y,
        sr=sample_rate,
        n_mfcc=n_mfccs,
        n_mels=n_mels,
        n_fft=n_fft,
        hop_length=hop_length,
        center=False,
    )


def _save_waveform_image(segment, sample_rate, out_path, title):
    t = np.arange(len(segment)) / float(sample_rate)
    fig, ax = plt.subplots(figsize=(cm_to_inch(FIG_W_CM), cm_to_inch(FIG_H_CM)), dpi=FIG_DPI)
    ax.plot(t, segment, linewidth=0.8)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.margins(x=0, y=0.05)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", pad_inches=0)
    plt.close(fig)


def _save_matrix_image(mat, out_path, title, ylabel, cmap="magma", enhance_contrast=False, interpolation="nearest"):
    fig, ax = plt.subplots(figsize=(cm_to_inch(FIG_W_CM), cm_to_inch(FIG_H_CM)), dpi=FIG_DPI)

    if enhance_contrast:
        finite = np.asarray(mat)[np.isfinite(mat)]
        if finite.size > 0:
            vmin = np.percentile(finite, MFCC_CONTRAST_LOW_PERCENTILE)
            vmax = np.percentile(finite, MFCC_CONTRAST_HIGH_PERCENTILE)
            if vmax <= vmin:
                vmin, vmax = float(np.min(finite)), float(np.max(finite))
            ax.imshow(mat, aspect="auto", origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, interpolation=interpolation)
        else:
            ax.imshow(mat, aspect="auto", origin="lower", cmap=cmap, interpolation=interpolation)
    else:
        ax.imshow(mat, aspect="auto", origin="lower", cmap=cmap, interpolation=interpolation)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

def _get_representative_paths_from_training_report(myUDH_obj, classes_list):
    """
    Select one representative acquisition path per class from myUDH.last_file_segment_report.
    This is the same acquisition list used by get_data_mfccs(), so it avoids folder-name guessing.
    """
    if not hasattr(myUDH_obj, "last_file_segment_report") or myUDH_obj.last_file_segment_report is None:
        raise RuntimeError(
            "myUDH.last_file_segment_report is missing. Run `mfccsTrain, labelsTrain, mfccsTest, labelsTest = myUDH.get_data_mfccs()` first."
        )

    report = myUDH_obj.last_file_segment_report.copy()
    if report.empty:
        raise RuntimeError(
            "myUDH.last_file_segment_report is empty. Run the data loading cell first and check whether acquisitions were read successfully."
        )

    required_cols = {"label", "acq_path", "status"}
    missing_cols = required_cols - set(report.columns)
    if missing_cols:
        raise RuntimeError(f"file_segment_report is missing columns: {missing_cols}")

    # Prefer valid rows that generated at least one segment.
    ok = report.copy()
    ok["label_str"] = ok["label"].astype(str)
    ok["status_str"] = ok["status"].astype(str).str.lower()
    if "segments" in ok.columns:
        ok["segments_int"] = ok["segments"].fillna(0).astype(int)
        ok = ok[(ok["status_str"] == "ok") & (ok["segments_int"] > 0)]
    else:
        ok = ok[ok["status_str"] == "ok"]

    out = {}
    for cls in classes_list:
        cls_str = str(cls)
        rows = ok[ok["label_str"] == cls_str]
        if rows.empty:
            rows = ok[ok["label_str"].str.lower() == cls_str.lower()]
        if rows.empty:
            out[cls_str] = None
        else:
            out[cls_str] = str(rows.iloc[0]["acq_path"])
    return out, report


def _draw_stack(ax, x, y, w, h, depth=4, dx=0.16, dy=0.11, facecolor="#bdd4ff", edgecolor="#7a94c9", label=""):
    for i in range(depth):
        rect = Rectangle((x + i * dx, y + i * dy), w, h, facecolor=facecolor, edgecolor=edgecolor, linewidth=1.0, alpha=0.55)
        ax.add_patch(rect)
    ax.text(x + w / 2 + depth * dx / 2, y + h / 2 + depth * dy / 2, label, ha="center", va="center", fontsize=9)


def _draw_fc_column(ax, x, y0, n_nodes, height=2.6, radius=0.07, color="#9fd6c2", label=""):
    ys = np.linspace(y0, y0 + height, n_nodes)
    pts = []
    for yy in ys:
        circ = plt.Circle((x, yy), radius, facecolor=color, edgecolor="gray", linewidth=0.8)
        ax.add_patch(circ)
        pts.append((x, yy))
    ax.text(x, y0 + height + 0.22, label, ha="center", va="bottom", fontsize=9)
    return pts


def _connect_columns(ax, pts1, pts2, color="gray", alpha=0.7, lw=0.5):
    idx1 = np.linspace(0, len(pts1) - 1, min(5, len(pts1))).astype(int)
    idx2 = np.linspace(0, len(pts2) - 1, min(5, len(pts2))).astype(int)
    for i in idx1:
        for j in idx2:
            (x1, y1), (x2, y2) = pts1[i], pts2[j]
            ax.plot([x1, x2], [y1, y2], color=color, linewidth=lw, alpha=alpha)


def _draw_arrow(ax, x1, y1, x2, y2):
    arr = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=12, linewidth=1.2, color="dimgray")
    ax.add_patch(arr)


def save_3d_model_framework(out_png, out_pdf=None):
    fig, ax = plt.subplots(figsize=(16, 7), dpi=300)
    ax.set_xlim(0, 18)
    ax.set_ylim(0, 8)
    ax.axis("off")

    _draw_stack(ax, 0.45, 5.2, 1.3, 0.85, depth=4, label="Waves\n(1.92 s, 16 kHz)")
    _draw_arrow(ax, 2.0, 5.7, 3.0, 5.7)
    _draw_stack(ax, 3.1, 4.95, 1.55, 1.2, depth=5, facecolor="#d9d0ff", edgecolor="#9b8fd1", label="Log-Mel\n32×60")
    _draw_arrow(ax, 4.9, 5.7, 5.9, 5.7)
    _draw_stack(ax, 6.0, 4.95, 1.55, 1.2, depth=5, facecolor="#dbeaf7", edgecolor="#8bb0c9", label="MFCC\n32×60")
    _draw_arrow(ax, 7.8, 5.7, 8.7, 5.7)
    _draw_stack(ax, 8.8, 4.95, 1.45, 1.2, depth=4, facecolor="#cce8df", edgecolor="#82b8a7", label="Lambda\nremove C0\n60×31×1")

    _draw_arrow(ax, 2.0, 4.9, 2.0, 3.75)
    _draw_arrow(ax, 2.0, 3.75, 3.0, 3.75)
    _draw_stack(ax, 3.1, 3.0, 1.55, 1.05, depth=5, label="Conv1\n3×3\n(16)")
    ax.text(4.0, 2.65, "MaxPool\n2×2", ha="center", va="top", fontsize=9)

    _draw_arrow(ax, 4.9, 3.55, 6.0, 3.55)
    _draw_stack(ax, 6.1, 3.0, 1.55, 1.05, depth=5, label="Conv2\n3×3\n(24)")
    ax.text(7.0, 2.65, "MaxPool\n2×2", ha="center", va="top", fontsize=9)

    _draw_arrow(ax, 7.9, 3.55, 9.0, 3.55)
    _draw_stack(ax, 9.1, 3.0, 0.6, 1.05, depth=4, facecolor="#e1f0e8", edgecolor="#8cb59d", label="Flatten")
    _draw_arrow(ax, 10.0, 3.55, 10.9, 3.55)

    pts1 = _draw_fc_column(ax, 11.35, 2.2, 8, label="Dense\n32")
    pts2 = _draw_fc_column(ax, 13.25, 2.35, 5, label="Dropout\n0.20")
    pts3 = _draw_fc_column(ax, 15.1, 2.15, 7, label="Softmax\n11")
    _connect_columns(ax, pts1, pts2)
    _connect_columns(ax, pts2, pts3)
    _draw_arrow(ax, 15.7, 3.55, 16.75, 3.55)

    out_box = Rectangle((16.85, 2.55), 1.05, 2.1, facecolor="#f0d8e1", edgecolor="#c19aa8", linewidth=1.2)
    ax.add_patch(out_box)
    ax.text(17.38, 4.28, "Output\n11 classes", ha="center", va="center", fontsize=9)
    symbols = ["□", "○", "△", "◇", "☆", "⬟", "▽", "⬢", "◁", "▷", "●"]
    for i, sym in enumerate(symbols):
        row = i % 4
        col = i // 4
        ax.text(17.02 + 0.25 * col, 3.9 - 0.33 * row, sym, fontsize=12, ha="center", va="center")

    ax.text(0.35, 7.35, "(a)", fontsize=12, fontweight="bold")
    ax.text(0.7, 7.35, "11-class ultrasound command recognition framework", fontsize=12)

    fig.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    if out_pdf is not None:
        fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)


def export_class_visuals_fixed(myUDH_obj, classes_list, export_root, sensor_name="imp23absu", sensor_type="mic", sample_start=0):
    export_root = Path(export_root)
    wave_dir = export_root / "waveforms_11classes"
    logmel_dir = export_root / "logmel_11classes"
    mfcc_dir = export_root / "mfcc_11classes"
    arch_dir = export_root / "architecture"
    for d in [wave_dir, logmel_dir, mfcc_dir, arch_dir]:
        _ensure_dir(d)

    representative_paths, report = _get_representative_paths_from_training_report(myUDH_obj, classes_list)

    rows = []
    for label in classes_list:
        label = str(label)
        acq_path = representative_paths.get(label)
        if acq_path is None:
            print(f"[WARN] No representative path in training report for class: {label}")
            rows.append({"label": label, "acq_path": None, "status": "missing_in_training_report"})
            continue

        raw = myUDH_obj.read_hsd_ultrasound(acq_path, sensor_name, sensor_type, sample_start)
        if raw is None or len(raw) == 0:
            print(f"[WARN] Unable to read representative acquisition: {label} | {acq_path}")
            rows.append({"label": label, "acq_path": acq_path, "status": "read_failed"})
            continue

        segment = _pad_or_crop(raw, myUDH_obj.frame_len).astype(np.float32)
        logmel = _compute_logmel(segment, myUDH_obj.sample_rate, myUDH_obj.nFFT, myUDH_obj.hop_length, myUDH_obj.n_mels)
        mfcc = _compute_mfcc(segment, myUDH_obj.sample_rate, myUDH_obj.nFFT, myUDH_obj.hop_length, myUDH_obj.n_mels, myUDH_obj.n_mfccs)

        stem = _safe_filename(label)
        _save_waveform_image(segment, myUDH_obj.sample_rate, wave_dir / f"{stem}_wave.png", f"{label} - waveform")
        _save_matrix_image(logmel, logmel_dir / f"{stem}_logmel.png", f"{label} - Log-Mel", "Mel bin")
        _save_matrix_image(mfcc, mfcc_dir / f"{stem}_mfcc.png", f"{label} - MFCC", "Coeff", cmap=MFCC_HIGH_CONTRAST_CMAP, enhance_contrast=True, interpolation="bicubic")

        rows.append({
            "label": label,
            "acq_path": acq_path,
            "segment_len": len(segment),
            "logmel_shape": tuple(logmel.shape),
            "mfcc_shape": tuple(mfcc.shape),
            "status": "ok",
        })
        print(f"[OK] {label}: {acq_path}")

    save_3d_model_framework(arch_dir / "model_framework_3d.png", arch_dir / "model_framework_3d.pdf")

    try:
        import pandas as pd
        summary_df = pd.DataFrame(rows)
        summary_df.to_csv(export_root / "visual_export_summary.csv", index=False, encoding="utf-8-sig")
        report.to_csv(export_root / "source_file_segment_report_used.csv", index=False, encoding="utf-8-sig")
        display(summary_df)
    except Exception as e:
        summary_df = rows
        print("[WARN] Could not create summary dataframe:", e)
        print(rows)

    print("\n[DONE] Fixed visual export finished.")
    print("Export root:", export_root)
    print("Waveforms  :", wave_dir)
    print("Log-Mel    :", logmel_dir)
    print("MFCC       :", mfcc_dir)
    print("Arch       :", arch_dir)
    return str(export_root), summary_df


# Run the fixed export.
# Important: run the data-loading cell first so myUDH.last_file_segment_report contains all 11 classes.
visual_export_root_fixed = os.path.join(resultDir, "visual_exports_fixed") if "resultDir" in globals() else os.path.join(".", "results_visual_exports_fixed")
visual_export_root_fixed, visual_export_summary_fixed = export_class_visuals_fixed(
    myUDH_obj=myUDH,
    classes_list=classes,
    export_root=visual_export_root_fixed,
    sensor_name=sensor_name,
    sensor_type=sensor_type,
    sample_start=sample_start,
)


## Added export: accuracy/loss, ROC, and t-SNE panels

This block generates paper-style panels similar to the referenced Figure 4 and saves the raw plot data as CSV.

Outputs are saved under:

```python
resultDir/performance_tsne_exports/
```


In [ ]:

# ============================================================
# Added export block: accuracy/loss, ROC, t-SNE panels + CSV
# ============================================================
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import tensorflow as tf


PERF_TSNE_DIR = Path(resultDir) / "performance_tsne_exports" if "resultDir" in globals() else Path("./performance_tsne_exports")
PERF_TSNE_DIR.mkdir(parents=True, exist_ok=True)

TSNE_MAX_SAMPLES_PER_CLASS = 80
TSNE_RANDOM_STATE = 611
TSNE_RAW_DOWNSAMPLE_POINTS = 256
FIG_DPI = 300


def _as_class_indices(y):
    y = np.asarray(y)
    if y.ndim > 1:
        return np.argmax(y, axis=1)
    return y.astype(int)


def _compute_tsne(features, random_state=611):
    features = np.asarray(features, dtype=np.float32)
    features = np.nan_to_num(features, copy=False)
    features = StandardScaler().fit_transform(features)

    if features.shape[1] > 50 and features.shape[0] > 5:
        n_comp = min(50, features.shape[0] - 1, features.shape[1])
        features = PCA(n_components=n_comp, random_state=random_state).fit_transform(features)

    perplexity = min(30, max(5, (features.shape[0] - 1) // 3))
    emb = TSNE(
        n_components=2,
        random_state=random_state,
        init="pca",
        learning_rate="auto",
        perplexity=perplexity,
    ).fit_transform(features)
    return emb


def export_accuracy_loss_curve(history_obj, out_dir=PERF_TSNE_DIR):
    if history_obj is None or not hasattr(history_obj, "history"):
        raise RuntimeError("Keras History object `history` was not found. Run model.fit(...) first.")

    hist = history_obj.history
    df = pd.DataFrame(hist)
    df.insert(0, "epoch", np.arange(len(df)))
    csv_path = out_dir / "accuracy_loss_history.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    acc_key = "accuracy" if "accuracy" in hist else ("acc" if "acc" in hist else None)
    val_acc_key = "val_accuracy" if "val_accuracy" in hist else ("val_acc" if "val_acc" in hist else None)

    fig, ax1 = plt.subplots(figsize=(3.0, 2.3), dpi=FIG_DPI)
    ax2 = ax1.twinx()

    if acc_key:
        ax1.plot(df["epoch"], df[acc_key], linewidth=1.5, label="Accuracy")
    if val_acc_key:
        ax1.plot(df["epoch"], df[val_acc_key], linewidth=1.0, linestyle="--", label="Val. Acc.")
    if "loss" in df:
        ax2.plot(df["epoch"], df["loss"], linewidth=1.2, label="Loss")
    if "val_loss" in df:
        ax2.plot(df["epoch"], df["val_loss"], linewidth=1.0, linestyle="--", label="Val. Loss")

    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Accuracy")
    ax2.set_ylabel("Loss")
    ax1.set_title("Accuracy & Loss curve", fontsize=9)

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, fontsize=6, loc="center right", frameon=False)

    fig.tight_layout()
    png_path = out_dir / "accuracy_loss_curve.png"
    pdf_path = out_dir / "accuracy_loss_curve.pdf"
    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    print("[OK] Saved:", csv_path)
    print("[OK] Saved:", png_path)


def export_multiclass_roc(model_obj, x_test, y_test, classes_list, out_dir=PERF_TSNE_DIR):
    y_true_idx = _as_class_indices(y_test)
    n_classes = len(classes_list)

    y_score = model_obj.predict(x_test, verbose=0)
    y_bin = label_binarize(y_true_idx, classes=np.arange(n_classes))

    rows = []
    auc_rows = []
    fig, ax = plt.subplots(figsize=(3.0, 2.3), dpi=FIG_DPI)

    for i, cls in enumerate(classes_list):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, thr = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        auc_rows.append({"class_index": i, "class_name": cls, "auc": roc_auc})
        for fp, tp, th in zip(fpr, tpr, thr):
            rows.append({
                "class_index": i,
                "class_name": cls,
                "fpr": fp,
                "tpr": tp,
                "threshold": th,
                "auc": roc_auc,
            })
        ax.plot(fpr, tpr, linewidth=1.0, label=f"{cls}(AUC={roc_auc:.2f})")

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=0.8)
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title("ROC curve", fontsize=9)
    ax.legend(fontsize=4.8, loc="lower right", frameon=False)
    fig.tight_layout()

    roc_csv = out_dir / "roc_curve_data.csv"
    auc_csv = out_dir / "roc_auc_summary.csv"
    pd.DataFrame(rows).to_csv(roc_csv, index=False, encoding="utf-8-sig")
    pd.DataFrame(auc_rows).to_csv(auc_csv, index=False, encoding="utf-8-sig")

    png_path = out_dir / "roc_curve.png"
    pdf_path = out_dir / "roc_curve.pdf"
    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    print("[OK] Saved:", roc_csv)
    print("[OK] Saved:", png_path)


def _select_balanced_indices(y_idx, max_per_class=80, random_state=611):
    rng = np.random.default_rng(random_state)
    selected = []
    for c in np.unique(y_idx):
        inds = np.where(y_idx == c)[0]
        if len(inds) > max_per_class:
            inds = rng.choice(inds, size=max_per_class, replace=False)
        selected.extend(list(inds))
    selected = np.array(selected, dtype=int)
    rng.shuffle(selected)
    return selected


def _plot_tsne_panel(tsne_df, title, out_png, out_pdf=None, classes_list=None):
    # 防止 concat 后出现重复列名导致 pandas reindex 报错
    tsne_df = tsne_df.loc[:, ~tsne_df.columns.duplicated()].copy()
    markers = ["o", "s", "^", "v", "D", "*", "P", "X", "<", ">", "h", "p"]
    fig, ax = plt.subplots(figsize=(3.0, 2.3), dpi=FIG_DPI)

    class_names = classes_list if classes_list is not None else sorted(tsne_df["class_name"].unique())
    for i, cls in enumerate(class_names):
        sub = tsne_df[tsne_df["class_name"] == cls]
        if len(sub) == 0:
            continue
        ax.scatter(
            sub["tsne_1"],
            sub["tsne_2"],
            s=12,
            marker=markers[i % len(markers)],
            alpha=0.75,
            linewidths=0.2,
            label=cls,
        )

    ax.set_title(title, fontsize=9)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.tick_params(labelsize=6)
    ax.legend(fontsize=4.8, frameon=False, loc="best", ncol=1)
    fig.tight_layout()
    fig.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    if out_pdf is not None:
        fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)


def _collect_raw_and_logmel_features_from_training_report(myUDH_obj, classes_list, max_per_class=80):
    if not hasattr(myUDH_obj, "last_file_segment_report") or myUDH_obj.last_file_segment_report is None:
        raise RuntimeError("myUDH.last_file_segment_report not found. Run myUDH.get_data_mfccs() first.")

    report = myUDH_obj.last_file_segment_report.copy()
    report = report[report["status"].astype(str).str.lower() == "ok"].copy()

    raw_features = []
    logmel_features = []
    labels = []
    source_rows = []

    for class_index, label in enumerate(classes_list):
        rows = report[report["label"].astype(str) == str(label)]
        if len(rows) == 0:
            print(f"[WARN] No training report row for class: {label}")
            continue

        collected = 0
        for _, row in rows.iterrows():
            if collected >= max_per_class:
                break

            acq_path = row.get("acq_path", None)
            if not acq_path:
                continue

            data = myUDH_obj.read_hsd_ultrasound(
                acq_path,
                myUDH_obj.sensor_name,
                myUDH_obj.sensor_type,
                myUDH_obj.offset,
            )
            if data is None or len(data) < myUDH_obj.frame_len:
                continue

            segs = myUDH_obj.get_data_segments(data, seqLen=myUDH_obj.frame_len, seqStep=myUDH_obj.frame_len)
            if segs is None or len(segs) == 0:
                continue

            for k in range(segs.shape[0]):
                if collected >= max_per_class:
                    break

                seg = segs[k].astype(np.float32)

                idx = np.linspace(0, len(seg) - 1, TSNE_RAW_DOWNSAMPLE_POINTS).astype(int)
                raw_feat = seg[idx]
                raw_feat = raw_feat - np.mean(raw_feat)
                std = np.std(raw_feat)
                if std > 1e-12:
                    raw_feat = raw_feat / std

                y = seg.copy()
                if y.size and np.max(np.abs(y)) > 1.0:
                    y = y / 32768.0
                mel = lb.feature.melspectrogram(
                    y=y,
                    sr=myUDH_obj.sample_rate,
                    n_fft=myUDH_obj.nFFT,
                    hop_length=myUDH_obj.hop_length,
                    win_length=myUDH_obj.nFFT,
                    n_mels=myUDH_obj.n_mels,
                    center=False,
                    power=2.0,
                )
                logmel = lb.power_to_db(mel, ref=np.max)

                raw_features.append(raw_feat)
                logmel_features.append(logmel.flatten())
                labels.append(class_index)
                source_rows.append({
                    "class_index": class_index,
                    "class_name": label,
                    "acq_path": acq_path,
                    "segment_index": k,
                })
                collected += 1

    return np.asarray(raw_features), np.asarray(logmel_features), np.asarray(labels), pd.DataFrame(source_rows)


def _get_fc_feature_model(model_obj):
    candidate_layer = None
    for layer in model_obj.layers:
        if isinstance(layer, tf.keras.layers.Dense) and getattr(layer, "units", None) == 32:
            candidate_layer = layer

    if candidate_layer is None:
        for layer in reversed(model_obj.layers):
            if isinstance(layer, tf.keras.layers.Dense):
                candidate_layer = layer
                break

    if candidate_layer is None:
        raise RuntimeError("No Dense layer found for FC feature extraction.")

    return tf.keras.Model(inputs=model_obj.input, outputs=candidate_layer.output), candidate_layer.name


def export_tsne_panels(model_obj, x_data, y_data, classes_list, myUDH_obj=None, out_dir=PERF_TSNE_DIR):
    y_idx_all = _as_class_indices(y_data)
    selected = _select_balanced_indices(y_idx_all, max_per_class=TSNE_MAX_SAMPLES_PER_CLASS, random_state=TSNE_RANDOM_STATE)
    x_sel = x_data[selected]
    y_sel = y_idx_all[selected]

    # ---- MFCC input feature clustering (after MFCC, before CNN) ----
    mfcc_features = np.asarray(x_sel).reshape((x_sel.shape[0], -1))
    mfcc_emb = _compute_tsne(mfcc_features, random_state=TSNE_RANDOM_STATE)
    mfcc_df = pd.DataFrame({
        "tsne_1": mfcc_emb[:, 0],
        "tsne_2": mfcc_emb[:, 1],
        "class_index": y_sel,
        "class_name": [classes_list[i] for i in y_sel],
        "source": "MFCC input features",
    })
    mfcc_csv = out_dir / "tsne_mfcc_features_data.csv"
    mfcc_df.to_csv(mfcc_csv, index=False, encoding="utf-8-sig")
    _plot_tsne_panel(mfcc_df, "MFCC features", out_dir / "tsne_mfcc_features.png", out_dir / "tsne_mfcc_features.pdf", classes_list=classes_list)

    # ---- FC-layer feature clustering ----
    fc_model, fc_layer_name = _get_fc_feature_model(model_obj)
    fc_features = fc_model.predict(x_sel, verbose=0)
    fc_features = fc_features.reshape((fc_features.shape[0], -1))

    fc_emb = _compute_tsne(fc_features, random_state=TSNE_RANDOM_STATE)
    fc_df = pd.DataFrame({
        "tsne_1": fc_emb[:, 0],
        "tsne_2": fc_emb[:, 1],
        "class_index": y_sel,
        "class_name": [classes_list[i] for i in y_sel],
        "source": f"FC feature layer: {fc_layer_name}",
    })
    fc_csv = out_dir / "tsne_fc_layer_data.csv"
    fc_df.to_csv(fc_csv, index=False, encoding="utf-8-sig")
    _plot_tsne_panel(fc_df, "FC layer features", out_dir / "tsne_fc_layer.png", out_dir / "tsne_fc_layer.pdf", classes_list=classes_list)

    raw_csv = None
    logmel_csv = None

    if myUDH_obj is not None:
        raw_features, logmel_features, raw_labels, source_df = _collect_raw_and_logmel_features_from_training_report(
            myUDH_obj,
            classes_list,
            max_per_class=TSNE_MAX_SAMPLES_PER_CLASS,
        )

        if raw_features.shape[0] > 2:
            raw_emb = _compute_tsne(raw_features, random_state=TSNE_RANDOM_STATE)
            raw_df = pd.DataFrame({
                "tsne_1": raw_emb[:, 0],
                "tsne_2": raw_emb[:, 1],
                "class_index": raw_labels,
                "class_name": [classes_list[i] for i in raw_labels],
            })
            raw_source_df = source_df.reset_index(drop=True).drop(columns=['class_index', 'class_name'], errors='ignore')
            raw_df = pd.concat([raw_df.reset_index(drop=True), raw_source_df], axis=1)
            raw_csv = out_dir / "tsne_original_waves_data.csv"
            raw_df.to_csv(raw_csv, index=False, encoding="utf-8-sig")
            _plot_tsne_panel(raw_df, "Original waves", out_dir / "tsne_original_waves.png", out_dir / "tsne_original_waves.pdf", classes_list=classes_list)

        if logmel_features.shape[0] > 2:
            logmel_emb = _compute_tsne(logmel_features, random_state=TSNE_RANDOM_STATE)
            logmel_df = pd.DataFrame({
                "tsne_1": logmel_emb[:, 0],
                "tsne_2": logmel_emb[:, 1],
                "class_index": raw_labels,
                "class_name": [classes_list[i] for i in raw_labels],
            })
            logmel_source_df = source_df.reset_index(drop=True).drop(columns=['class_index', 'class_name'], errors='ignore')
            logmel_df = pd.concat([logmel_df.reset_index(drop=True), logmel_source_df], axis=1)
            logmel_csv = out_dir / "tsne_logmel_spectrogram_data.csv"
            logmel_df.to_csv(logmel_csv, index=False, encoding="utf-8-sig")
            _plot_tsne_panel(logmel_df, "Spectrograms", out_dir / "tsne_logmel_spectrograms.png", out_dir / "tsne_logmel_spectrograms.pdf", classes_list=classes_list)

    print("[OK] Saved:", mfcc_csv)
    print("[OK] Saved:", fc_csv)
    if raw_csv:
        print("[OK] Saved:", raw_csv)
    if logmel_csv:
        print("[OK] Saved:", logmel_csv)


def export_combined_paper_style_panel(out_dir=PERF_TSNE_DIR):
    import matplotlib.image as mpimg

    panel_files = [
        ("(b) Accuracy & Loss curve", out_dir / "accuracy_loss_curve.png"),
        ("(c) ROC curve", out_dir / "roc_curve.png"),
        ("(e-I) Original waves", out_dir / "tsne_original_waves.png"),
        ("(e-II) Spectrograms", out_dir / "tsne_logmel_spectrograms.png"),
        ("(e-III) MFCC features", out_dir / "tsne_mfcc_features.png"),
        ("(e-IV) FC layer", out_dir / "tsne_fc_layer.png"),
    ]

    existing = [(title, path) for title, path in panel_files if path.exists()]
    if not existing:
        raise RuntimeError("No panel images found. Run the export functions first.")

    fig, axes = plt.subplots(1, len(existing), figsize=(3.0 * len(existing), 2.3), dpi=FIG_DPI)
    if len(existing) == 1:
        axes = [axes]

    for ax, (title, path) in zip(axes, existing):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(title, fontsize=8)
        ax.axis("off")

    fig.tight_layout()
    png_path = out_dir / "combined_accuracy_roc_tsne_panel.png"
    pdf_path = out_dir / "combined_accuracy_roc_tsne_panel.pdf"
    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    print("[OK] Saved:", png_path)


# -------------------------
# Run exports
# -------------------------
export_accuracy_loss_curve(history, out_dir=PERF_TSNE_DIR)
export_multiclass_roc(model, mfccsTest, labelsTest, classes, out_dir=PERF_TSNE_DIR)

export_tsne_panels(
    model_obj=model,
    x_data=mfccsTest,
    y_data=labelsTest,
    classes_list=classes,
    myUDH_obj=myUDH,
    out_dir=PERF_TSNE_DIR,
)

export_combined_paper_style_panel(out_dir=PERF_TSNE_DIR)

print("\n[DONE] Performance/ROC/t-SNE export finished.")
print("Output folder:", PERF_TSNE_DIR)
